In [1]:
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

# Load configuration
from real_state.config import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: OpenRouter
📁 Project root: d:\ai\bootcamp\mini-project-02\real_state


In [2]:
from real_state.application.chat_service import RagService

print("Chat services loaded from sevice layer")
print("📍 Location: context_engineering.application.chat_service")
print("\n🎯 Available services:")
print("   1. RAGService   - Standard RAG with modern LCEL")
print("   2. CAGService   - Cache-Augmented Generation (semantic)")
print("   3. CRAGService  - Corrective RAG with confidence scoring")
print("   4. CAGCache     - Semantic cache (FAQs + History)")

Chat services loaded from sevice layer
📍 Location: context_engineering.application.chat_service

🎯 Available services:
   1. RAGService   - Standard RAG with modern LCEL
   2. CAGService   - Cache-Augmented Generation (semantic)
   3. CRAGService  - Corrective RAG with confidence scoring
   4. CAGCache     - Semantic cache (FAQs + History)


In [3]:
from real_state.infrastructure.llm_provider import (
    get_default_embeddings,
    get_chat_llm

)
from real_state.infrastructure.db.vector_db import QdrantStorage


emb = get_default_embeddings()
llm = get_chat_llm(temperature=0)

print(f"✅ LLM initialized: {CHAT_MODEL}")
print(f"✅ Embeddings initialized: {EMBEDDING_MODEL}")
print(f"🌐 Provider: {PROVIDER}")

qdrant_client = QdrantStorage()

try:
    qdrant_client.collection_info(collection_name="prime_lands")
except:
    print("❌ Collection not found")




✅ LLM initialized: google/gemini-2.5-flash
✅ Embeddings initialized: openai/text-embedding-3-large
🌐 Provider: openrouter


In [4]:
rag_service = RagService(
    retriever = qdrant_client,
    llm = llm,
    k = TOP_K_RESULTS
)

print("✅ RAGService initialized")
print(f"🎯 Retrieval: top-{TOP_K_RESULTS} documents")

✅ RAGService initialized
🎯 Retrieval: top-4 documents


In [5]:
USER_QUERY = "how much SKYE BLOSSOM - KOTTAWA?"
print(f"🔍 Query: {USER_QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(USER_QUERY)

print(f"\n⏱️  Generation time: {result['generation_time']:.2f}s")
print(f"📄 Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")

🔍 Query: how much SKYE BLOSSOM - KOTTAWA?

GENERATING ANSWER WITH RAG SERVICE...

⏱️  Generation time: 5.78s
📄 Documents used: 4

ANSWER
**Key Facts**
*   SKYE BLOSSOM - KOTTAWA is a high-rise apartment project located in Kottawa [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en].
*   It is described as a rare investment opportunity and the only high-rise of its kind in the thriving suburb of Kottawa [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en].
*   The project enjoys exceptionally high rental demand [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en].

**Answer**
The price for SKYE BLOSSOM - KOTTAWA is 50,000,000 LKR [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en]. This project is nestled in the heart of Kottawa and is considered an investment opportunity due to its high rental demand [https://www.primelands.lk/apartment/SKYE-BLOSSOM-KOTTAWA/en].

**Additional Details**
*   **Pricing:** 50,000,000 LKR [https://www.primelands.lk/apart